In [ ]:
# =========================================================
# NIFTY PAPER TRADING ENGINE
# WEBHOOK -> ITM OPTION -> TP/SL TRACKING
# PORT = 5001
# MARKET TIME FILTER = 9:30 AM TO 3:00 PM
# =========================================================

from flask import Flask, request
from NorenRestApiPy.NorenApi import NorenApi
import yaml
import threading
import time

from datetime import datetime

# =========================================================
# API
# =========================================================

class ShoonyaApiPy(NorenApi):

    def __init__(self):

        super().__init__(
            host='https://trade.shoonya.com/NorenWClientWeb/',
            websocket='wss://trade.shoonya.com/NorenWSWeb/'
        )

api = ShoonyaApiPy()

# =========================================================
# LOGIN
# =========================================================

with open('cred.yml') as f:

    cred = yaml.load(f, Loader=yaml.FullLoader)

SuperToken = "466869ba14f23f0d303c383063951961f77ae233b5ee201e6b951ec9baebb2fb"

ret = api.set_session(
    cred['user'],
    cred['pwd'],
    SuperToken
)

print("\nLOGIN :", ret)

# =========================================================
# SETTINGS
# =========================================================

TP = 15

SL = -10

# =========================================================
# GLOBALS
# =========================================================

trade_running = False

pnl_array = []

live_data = {}

# =========================================================
# WEBSOCKET
# =========================================================

def open_callback():

    print("\n✅ WEBSOCKET CONNECTED")

def quote_update(message):

    try:

        token = message['tk']

        live_data[token] = message

    except:

        pass

api.start_websocket(
    order_update_callback=None,
    subscribe_callback=quote_update,
    socket_open_callback=open_callback
)

# =========================================================
# GET OPTION
# =========================================================

def get_option(signal):

    # =====================================================
    # NIFTY SPOT
    # =====================================================

    nifty_quote = api.get_quotes(
        exchange="NSE",
        token="26000"
    )

    nifty_ltp = float(nifty_quote['lp'])

    print(f"\nNIFTY SPOT : {nifty_ltp}")

    # =====================================================
    # STRIKES
    # =====================================================

    ce_strike = (int(nifty_ltp / 50) * 50) + 100

    pe_strike = (int(nifty_ltp / 50) * 50) - 100
    
    print(f"ITM CE : {ce_strike}")
    print(f"ITM PE : {pe_strike}")

    # =====================================================
    # FIND FUTURE SYMBOL
    # =====================================================

    search = api.searchscrip(
        exchange='NFO',
        searchtext='NIFTY'
    )

    future_symbol = None

    for item in search['values']:

        tsym = item['tsym']

        if (
            tsym.startswith('NIFTY')
            and tsym.endswith('F')
            and 'BANKNIFTY' not in tsym
            and 'FINNIFTY' not in tsym
            and 'MIDCPNIFTY' not in tsym
            and 'NIFTYNXT50' not in tsym
        ):

            future_symbol = tsym
            break

    print(f"\nFUTURE : {future_symbol}")

    # =====================================================
    # OPTION CHAIN
    # =====================================================

    chain = api.get_option_chain(
        exchange='NFO',
        tradingsymbol=future_symbol,
        strikeprice=ce_strike,
        count=5
    )

    ce_token = None
    pe_token = None

    ce_symbol = None
    pe_symbol = None

    for item in chain['values']:

        strike = float(item['strprc'])

        opt_type = item['optt'].upper()

        # BUY -> ITM CE

        if strike == ce_strike and opt_type in ['CE', 'C']:

            ce_token = item['token']
            ce_symbol = item['tsym']

        # SELL -> ITM PE

        elif strike == pe_strike and opt_type in ['PE', 'P']:

            pe_token = item['token']
            pe_symbol = item['tsym']

    # =====================================================
    # FINAL OPTION
    # =====================================================

    if signal == "BUY":

        return ce_symbol, ce_token

    else:

        return pe_symbol, pe_token

# =========================================================
# TRADE MONITOR
# =========================================================

def monitor_trade(token, symbol, entry_price):

    global trade_running
    global pnl_array

    print("\n===================================")
    print("TRADE MONITOR STARTED")
    print("===================================")

    print(f"SYMBOL : {symbol}")
    print(f"ENTRY : {entry_price}")

    while trade_running:

        try:

            # =========================================
            # LIVE LTP
            # =========================================

            if token in live_data:

                msg = live_data[token]

                if 'lp' in msg:

                    ltp = float(msg['lp'])

                else:

                    time.sleep(1)
                    continue

            else:

                q = api.get_quotes(
                    exchange='NFO',
                    token=token
                )

                ltp = float(q['lp'])

            # =========================================
            # DIFF
            # =========================================

            diff = round(ltp - entry_price, 2)

            print(
                f"LTP : {ltp} | "
                f"ENTRY : {entry_price} | "
                f"DIFF : {diff}"
            )

            # =========================================
            # TARGET
            # =========================================

            if diff >= TP:

                print("\n🎯 PROFIT BOOKED")

                pnl_array.append(diff)

                print("PNL ARRAY :", pnl_array)

                trade_running = False

                break

            # =========================================
            # STOPLOSS
            # =========================================

            elif diff <= SL:

                print("\n🛑 STOPLOSS HIT")

                pnl_array.append(diff)

                print("PNL ARRAY :", pnl_array)

                trade_running = False

                break

            time.sleep(1)

        except Exception as e:

            print("MONITOR ERROR :", e)

            time.sleep(1)

# =========================================================
# CREATE TRADE
# =========================================================

def create_trade(signal):

    global trade_running

    if trade_running:

        print("⚠ TRADE ALREADY RUNNING")
        return

    # =====================================================
    # GET OPTION
    # =====================================================

    symbol, token = get_option(signal)

    print(f"\nSELECTED : {symbol}")
    print(f"TOKEN : {token}")

    # =====================================================
    # SUBSCRIBE
    # =====================================================

    api.subscribe(f"NFO|{token}")

    time.sleep(2)

    # =====================================================
    # ENTRY PRICE
    # =====================================================

    q = api.get_quotes(
        exchange='NFO',
        token=token
    )

    entry_price = float(q['lp'])

    print(f"\nENTRY PRICE : {entry_price}")

    trade_running = True

    # =====================================================
    # START MONITOR
    # =====================================================

    threading.Thread(
        target=monitor_trade,
        args=(token, symbol, entry_price),
        daemon=True
    ).start()

# =========================================================
# FLASK
# =========================================================

app = Flask(__name__)

# =========================================================
# WEBHOOK
# =========================================================

@app.route('/webhook', methods=['POST'])

def webhook():

    # =====================================================
    # TIME FILTER
    # =====================================================

    now = datetime.now().time()

    market_start = datetime.strptime("09:30", "%H:%M").time()

    market_end = datetime.strptime("15:00", "%H:%M").time()

    if not (market_start <= now <= market_end):

        print("\n===================================")
        print("⚠ SIGNAL IGNORED")
        print("OUTSIDE MARKET TIME")
        print("===================================")

        return "MARKET CLOSED"

    # =====================================================
    # WEBHOOK DATA
    # =====================================================

    data = request.json

    print("\n===================================")
    print("WEBHOOK RECEIVED")
    print("===================================")

    print(data)
    
    signal = data.get("signal").upper()

    create_trade(signal)

    return "OK"

# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    print("\n===================================")
    print("🚀 PAPER ENGINE STARTED")
    print("===================================")

    print("\n📡 WAITING FOR WEBHOOKS...")

    app.run(
        host='0.0.0.0',
        port=5001
    )


LOGIN : True

🚀 PAPER ENGINE STARTED

📡 WAITING FOR WEBHOOKS...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://192.168.1.5:5001
Press CTRL+C to quit



✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED


192.168.1.5 - - [22/May/2026 11:16:50] "GET / HTTP/1.1" 404 -
192.168.1.5 - - [22/May/2026 11:16:52] "GET /favicon.ico HTTP/1.1" 404 -
192.168.1.5 - - [22/May/2026 11:17:22] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [22/May/2026 11:17:34] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [22/May/2026 11:17:35] "GET /favicon.ico HTTP/1.1" 404 -



✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKET CONNECTED

✅ WEBSOCKE